# Линейная регрессия на синтетических данных (продолжение)

Продолжение ноутбука `Lect03_LR_synthetic_data.ipynb`: там мы строили линейную регрессию на простых данных, а здесь на тех же идеях разбираем, как усложнение модели влияет на качество.

Рассматриваются следующие вопросы:
- Порождение новых признаков
- Гиперпараметры модели
- Проблема переобучения модели
- Кросс-валидация, тестовая и тренировочные выборки
- Масштабирование данных (стандартизация) 

Это «анимационная» версия ноутбука `Lect04_LRext_synthetic_data.ipynb`: перебор степени полинома оформлен здесь как анимация (последняя ячейка), а данные задаются равномерной сеткой.

### Полезные ссылки:
- <a href="https://numpy.org/">Numpy documentation</a><br />
- <a href="https://pandas.pydata.org/">Pandas documentation</a><br />
- <a href="https://matplotlib.org/">Matplotlib documentation</a><br />
- <a href="https://scikit-learn.org/stable/">sklearn documentation</a><br />
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)<br />
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [метрики регрессии](https://scikit-learn.org/stable/modules/model_evaluation.html#regression-metrics)<br />
- [matplotlib.animation.FuncAnimation](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html), [сохранение анимаций](https://matplotlib.org/stable/api/animation_api.html#writer-classes)

## Импорт библиотек

Числовые и табличные инструменты, графика, а также всё нужное из scikit-learn: линейная регрессия, полиномиальные признаки, масштабирование, разделение выборок и метрики. Дополнительно настраиваем отрисовку: `%matplotlib inline` показывает графики прямо в ноутбуке, а `dpi` задаёт их разрешение (для анимации это влияет и на размер gif-файла).

**Инструменты:** [NumPy](https://numpy.org/doc/stable/), [pandas](https://pandas.pydata.org/docs/), [matplotlib.pyplot](https://matplotlib.org/stable/api/pyplot_summary.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [sklearn.metrics](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

%matplotlib inline 
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.dpi'] = 150

## Генерация данных 

Порождаем синтетические данные с зависимостью y от x в форме многочлена 3-й степени. Для этого используем модель препроцессинга [PolynomialFeatures](https://scikit-learn.org/dev/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) библиотеки sklearn. 

Внутри `generate_data`: `X` — 33 точки равномерной сетки от −10 до 10 с небольшим случайным разбросом (в статической версии ноутбука они случайные, здесь — по порядку), `X_poly` — матрица степеней $[1, x, x^2, x^3]$ (печатаем её, чтобы увидеть, как из одного столбца получается четыре), а `Y` считается по «истинным» коэффициентам `true_theta` с добавлением гауссова шума. В конце рисуем облако точек «X — Y».

**Инструменты:** [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [numpy.linspace](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html), [numpy.random.randn](https://numpy.org/doc/stable/reference/random/generated/numpy.random.randn.html), [numpy.dot](https://numpy.org/doc/stable/reference/generated/numpy.dot.html), [matplotlib.pyplot.plot](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.plot.html).

In [ ]:
def generate_data (sample_size = 40):
    
    #true_degee = 4; true_theta = [0, -2.5, -1.9, -0.019, 0.021]
    true_degee = 3; true_theta = [-50, -4, 1.9, 0.25]

    
    #X = np.random.rand(sample_size) * 20 - 10
    X = np.linspace(-10, 10, sample_size) + np.random.randn(sample_size)*0.25
    
    poly = PolynomialFeatures(true_degee)
    X_poly = poly.fit_transform(X.reshape(-1, 1))
    print('X shape: ', X.shape)
    print('X_poly shape: ', X_poly.shape)

    display(pd.DataFrame (X).head())
    display(pd.DataFrame (X_poly).head())

    Y = np.dot(X_poly, true_theta) + np.random.randn(sample_size) * 5 #+ np.random.randn(sample_size) * np.sqrt (np.abs(X)) * 2
    return X, Y

X, Y = generate_data (33)

plt.figure()
plt.plot(X, Y, '.k')
plt.xlabel('X')
plt.ylabel('Y')
plt.ylim([-100, 100])
plt.xlim([-10, 10])


## Создаем новый класс моделей машинного обучения - полиномиальную регрессию

Данная модель объединяет модель линейной регресии [LinearRegression](https://scikit-learn.org/1.5/modules/generated/sklearn.linear_model.LinearRegression.html), модель порождения полиномиальных признаков   [PolynomialFeatures](https://scikit-learn.org/dev/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) и, опционально, произвольную модель масштабирования данных.  

Модель имеет два **гиперпараметра**:
- Степень полинома
- Модель масштабирования данных (по умолчанию отсутствует)

Класс повторяет интерфейс sklearn: `fit` запоминает преобразование признаков и коэффициенты, `predict` считает прогноз. Замечание по учебному коду: в `predict` преобразование `PolynomialFeatures` настраивается заново на тех данных, на которых делается прогноз; в «боевом» коде преобразование сохраняют из `fit` (именно так поступает `Pipeline`), иначе результат зависит от выборки, на которой вызван `predict`.

**Инструменты:** [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html), [как писать свои оценки в стиле sklearn](https://scikit-learn.org/stable/developers/develop.html).

In [ ]:
class PolynomialRegression:
    def __init__(self, degree = 1, scaler = None):
        self.degree = degree
        self.regressor = LinearRegression()
        self.scaler = scaler () if scaler is not None else None
        
    def fit(self, X, Y):
        self.poly = PolynomialFeatures(self.degree)
        X_poly = self.poly.fit_transform(X.reshape(-1, 1))
        
        if self.scaler is not None:
            X_poly = self.scaler.fit_transform(X_poly)
        #display(pd.DataFrame(X_poly).describe())
        self.regressor.fit(X_poly, Y)
        
    def predict(self, X):
        X_poly = self.poly.fit_transform(X.reshape(-1, 1))
        if self.scaler is not None:
            X_poly = self.scaler.transform(X_poly)       
        return self.regressor.predict(X_poly)

## Обучение модели с вручную задаваемыми гиперпараметрами 

Изучаем, как ведет себя модель с различными гиперпараметрами (меняем степень полинома и модель масштабирования данных) и при различном размере набора данных.

Функция `evaluate_model` обучает переданную модель на всех данных, печатает RMSE и строит две картинки: слева — точки данных и предсказанная моделью кривая (по 10 000 точкам сетки), справа — диаграмма «Y предсказанное против Y наблюдённого» (чем ближе к диагонали, тем лучше).

Степень полинома 33 при трёх десятках наблюдений — заведомо «слишком гибкая» модель: она почти идеально проходит через точки, но между ними ведёт себя непредсказуемо. Это и есть переобучение, которое дальше будем измерять.

**Инструменты:** [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html), [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [r2_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html), [matplotlib.pyplot.subplots](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html).

In [ ]:
def evaluate_model (model, X, Y):
    model.fit(X, Y)
    Y_predicted = model.predict(X)
    rmse = np.sqrt(mean_squared_error(Y, Y_predicted))
    r2 = r2_score(Y, Y_predicted)
    
    print ('RMSE = ', rmse)
    X_smooth = np.linspace(X.min(), X.max(), 10000)
    Y_predicted_smooth = model.predict(X_smooth)

    plt.figure(figsize=(10,5))
    plt.subplot(1,2,2)
    plt.plot (Y, Y_predicted, 'ok', label = 'RMSE = %.2f, r2 = %.2f'%(rmse, r2))
    plt.axis('equal')
    plt.grid()
    plt.ylim([-100, 350])
    plt.xlim([-100, 350])
    plt.xlabel('Y наблюдённое')
    plt.ylabel('Y предсказанное моделью')
    plt.legend(loc='upper left')

    plt.subplot(1,2,1)
    plt.plot(X, Y, 'ok')
    plt.ylim([-100, 200])
    plt.plot(X_smooth, Y_predicted_smooth, '-r')   # предсказанная моделью кривая
    plt.xlabel('X (входной признак)')
    plt.ylabel('Y (отклик)')
    plt.suptitle('Degree = %d, scaler = %s, number of samples = %d'%(model.degree, model.scaler.__class__.__name__, Y.shape[0]))

#model = PolynomialRegression(30) 
model = PolynomialRegression(33, scaler=StandardScaler) 

evaluate_model (model, X, Y)

## Оптимизация ~~гиперпараметров~~ гиперпараметра методом конечного перебора

Подойдем к проблеме оптимизации гиперпараметра (степени полинома) формально и попробуем найти оптимальное значение. 

Перебираем степени от 0 до 34: для каждой обучаем модель на всех данных, считаем RMSE и складываем результат в список, затем отмечаем точку с минимальным RMSE.

Обратите внимание: качество здесь измеряется на тех же данных, на которых модель обучалась. Чем выше степень, тем гибче модель, и RMSE монотонно падает почти до нуля (поэтому ось значений логарифмическая). Такой «оптимум» — иллюзия, он ведёт к переобучению; именно поэтому нужна тестовая выборка (следующая ячейка).

**Инструменты:** [numpy.argmin](https://numpy.org/doc/stable/reference/generated/numpy.argmin.html), [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html), [matplotlib.pyplot.yscale](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.yscale.html).

In [ ]:
model_quality = []

degrees = range(0, 35)
for degree in degrees:
    model = PolynomialRegression(degree, scaler = StandardScaler)
    model.fit(X, Y)
    Y_predicted = model.predict(X)
    model_quality.append (np.sqrt(mean_squared_error(Y, Y_predicted)))

best_idx  = np.argmin(model_quality)

plt.figure(figsize=(10,5))
plt.plot (degrees, model_quality, '-k')
plt.plot (degrees[best_idx], model_quality[best_idx], 'ok')
plt.grid()
plt.xlabel('Степень полинома')
plt.ylabel('RMSE (обучающая выборка)')
plt.title('Переобучение: RMSE на обучающей выборке падает с ростом степени')

plt.yscale('log')
_ = plt.xticks(degrees)

## Оптимизация ~~гиперпараметров~~ гиперпараметра методом конечного перебора - **теперь с разделением на тренировочную и тестовую выборки**

Повторяем перебор, но теперь на каждой итерации данные заново делятся на тренировочную (75 %) и тестовую (25 %) части: модель обучается на первой, а RMSE считаются отдельно на обучающей и на тестовой выборках.

В этой (анимационной) версии ноутбука перебор оформлен как анимация `FuncAnimation`: один кадр — это одно случайное разбиение данных. На левом графике видны точки train/test и кривая лучшей по тесту модели, на правом — кривые RMSE и отмеченные минимумы; при каждом новом кадре картинка перестраивается. Готовый gif сохраняется в файл `animation.gif` в рабочей папке (он же игнорируется git).

**Инструменты:** [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html), [numpy.argmin](https://numpy.org/doc/stable/reference/generated/numpy.argmin.html), [matplotlib.animation.FuncAnimation](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.FuncAnimation.html), [Animation.save](https://matplotlib.org/stable/api/_as_gen/matplotlib.animation.Animation.html#matplotlib.animation.Animation.save), [numpy.linspace](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html).

In [ ]:
import matplotlib.animation as animation

plt.rcParams['savefig.dpi'] = 150
plt.rcParams['figure.dpi'] = 150

degrees = range(1, Y.shape[0]+1)


fig = plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
# p11, = plt.plot(x_train, y_train, 'ok', label = 'train')
# p12, = plt.plot(x_test, y_test, 'ob', label = 'test')
# p13, = plt.plot(X_smooth, Y_predicted_smooth, '-r', label = 'best model (test)')
p11, = plt.plot(0, 0, 'ok', label = 'train')
p12, = plt.plot(0, 0, 'ob', label = 'test')
p13, = plt.plot(0, 0, '-r', label = 'best model (test)')

plt.xlabel('X (входной признак)')
plt.ylabel('Y (отклик)')
plt.xlim([-10, 10])
plt.ylim([-100, 400])
plt.legend()

plt.subplot(1,2,2)
# p21, = plt.plot (degrees, rmse_train, '-k', label = 'train')
# p22, = plt.plot (degrees, rmse_test, '-b', label = 'test')
# p23, = plt.plot (degrees[best_idx_train], rmse_train[best_idx_train], 'ok')
# p24, = plt.plot (degrees[best_idx_test], rmse_test[best_idx_test], 'ob')
p21, = plt.plot (1, 1, '-k', label = 'train')
p22, = plt.plot (1, 1, '-b', label = 'test')
p23, = plt.plot (1, 1, 'ok')
p24, = plt.plot (1, 1, 'ob')
plt.xlim([0, degrees[-1]])


plt.grid()
plt.legend(loc='upper left')
plt.xlabel('Степень полинома')
plt.ylabel('RMSE')

plt.yscale('log')
plt.ylim([10**-10, 10**10])


def optimize_hyperparameter (i):

    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.25)

    models = []
    rmse_test = []
    rmse_train = []


    for degree in degrees:
        model = PolynomialRegression(degree, scaler = StandardScaler)
        
        model.fit(x_train, y_train)
        y_test_predicted = model.predict(x_test)
        y_train_predicted = model.predict(x_train)
        
        rmse_test.append (np.sqrt(mean_squared_error(y_test, y_test_predicted)))
        rmse_train.append (np.sqrt(mean_squared_error(y_train, y_train_predicted)))
        models.append(model)

    best_idx_test  = np.argmin(rmse_test)
    best_idx_train = np.argmin(rmse_train)

    model = models[best_idx_test]
    X_smooth = np.linspace(X.min(), X.max(), 10000)
    Y_predicted_smooth = model.predict(X_smooth)

    p11.set_data(x_train, y_train)
    p12.set_data(x_test, y_test)
    p13.set_data(X_smooth, Y_predicted_smooth)

    p21.set_data(degrees, rmse_train)
    p22.set_data(degrees, rmse_test)
    p23.set_data(degrees[best_idx_train], rmse_train[best_idx_train])
    p24.set_data(degrees[best_idx_test], rmse_test[best_idx_test])

    return p11, p12, p13, p21, p22, p23, p24


optimize_hyperparameter(0)    

ani = animation.FuncAnimation(
    fig,
    optimize_hyperparameter,
    interval=500,
    blit=False,  # blitting can't be used with Figure artists
    frames=range(0,10),
    repeat_delay=100,
)

ani.save('animation.gif') 